# 05 – Recommendation Engine

Demonstrate rule- and statistics-based recommendation logic and content calendar strategy generation.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from ml.preprocessing import load_and_clean
from backend.services.recommendation_service import _build_reasons, DAY_ORDER

df = load_and_clean('../data/sample/sample_posts.csv')
print(f'Loaded dataset with {len(df)} rows')

In [ ]:
# Calculate category averages
ct_avg = df.groupby('content_type')['engagement_rate'].mean().to_dict()
topic_avg = df.groupby('topic')['engagement_rate'].mean().to_dict()
day_avg = df.groupby('day_of_week')['engagement_rate'].mean().to_dict()
overall_avg = df['engagement_rate'].mean()

print('Overall average engagement rate:', round(overall_avg, 2), '%')
print('\nTop content types:', sorted(ct_avg.items(), key=lambda x: x[1], reverse=True))
print('\nTop topics:', sorted(topic_avg.items(), key=lambda x: x[1], reverse=True))

In [ ]:
# Generate Top 5 Recommendations
from itertools import product

content_types = df['content_type'].unique().tolist()
topics = df['topic'].unique().tolist()
days = [d for d in DAY_ORDER if d in day_avg]

combos = []
for ct, topic, day in product(content_types, topics, days):
    score = 0.40 * ct_avg.get(ct, overall_avg) + 0.35 * topic_avg.get(topic, overall_avg) + 0.25 * day_avg.get(day, overall_avg)
    combos.append((ct, topic, day, score))

combos.sort(key=lambda x: x[3], reverse=True)

rec_df = pd.DataFrame(combos[:5], columns=['Content Type', 'Topic', 'Day', 'Raw Score'])
rec_df['Normalized Score'] = (60 + 40 * (rec_df['Raw Score'] - combos[-1][3]) / (combos[0][3] - combos[-1][3])).round(1)
rec_df